# Neural networks with AAindex features

Train and evaluate the early binary classifiers on AAindex features. [Run configurations and curves](../../results/task1/model_selection/early_experiments/v2_aaindex/)


In [ ]:
import sys
sys.path.append('..')

In [ ]:
import torch.nn as nn
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from protlearn import features, preprocessing
from src.preprocessing import Task1Dataset, split_dataset
from sklearn.metrics import accuracy_score, matthews_corrcoef
from pathlib import Path
import matplotlib.pyplot as plt
import yaml

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Import dataset / sequences
df = pd.read_csv('../data/preprocessed/exp_proteins_c100_flens25-75_usamp_enz.tsv', sep='\t')

In [ ]:
standardize = 'zscore'
seq_list = list(df['Sequence'])
seq_list_aaindex = features.aaindex1(seq_list, standardize=standardize)[0]
X = torch.tensor(seq_list_aaindex, dtype=torch.float32)
y = torch.tensor(list(df['Enzyme']), dtype=torch.float32).unsqueeze(dim=1)
keys = list(df['Entry'])

In [ ]:
dataset = Task1Dataset(X, y, 'exp_proteins_c100_flens25-75_usamp_enz', keys)
train_val_set, test_set = split_dataset(dataset=dataset, train_size=0.8)
train_set, val_set = split_dataset(dataset=train_val_set, train_size=0.8)

In [ ]:
batch_size = 32
train_loader = DataLoader(dataset=train_set, batch_size=batch_size, shuffle=True)

In [ ]:
# Initialize simple model
activation = nn.LeakyReLU()
model = nn.Sequential(
     nn.Linear(553, 512, dtype=torch.float32),
     activation,
     nn.Dropout(0.5),
     nn.Linear(512, 369, dtype=torch.float32),
     activation,
     nn.Linear(369, 128, dtype=torch.float32),
     activation,
     nn.Dropout(0.2),
     nn.Linear(128, 1, dtype=torch.float32),
     nn.Sigmoid()
)

# Initialize hyperparameters
lr = 0.0001
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.BCELoss()
n_epochs = 200

In [ ]:
# Train model
train_losses, train_accs, train_mccs = list(), list(), list()
val_losses, val_accs, val_mccs = list(), list(), list()

for e in range(n_epochs):
    model.train()
    total_loss, total_acc, total_mcc = 0, 0, 0
    for feats, labels, _ in train_loader:
        feats, labels = feats.to(device=device), labels.to(device=device)
        optimizer.zero_grad()
        outputs = model(feats)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        with torch.no_grad():
            outputs_rounded = torch.round(outputs)
            total_acc += accuracy_score(labels.cpu(), outputs_rounded.cpu())
            total_mcc += matthews_corrcoef(labels.cpu(), outputs_rounded.cpu())

    avg_train_loss = round(total_loss / len(train_loader), 4)
    avg_train_acc = round(total_acc / len(train_loader), 4)
    avg_train_mcc = round(total_mcc / len(train_loader), 4)
    train_losses.append(avg_train_loss)
    train_accs.append(avg_train_acc)
    train_mccs.append(avg_train_mcc)

    model.eval()
    with torch.no_grad():
        outputs = model(val_set.X)
        outputs_rounded = torch.round(outputs)
        val_loss = criterion(outputs, val_set.y).item()
        val_acc = accuracy_score(val_set.y, outputs_rounded.cpu())
        val_mcc = matthews_corrcoef(val_set.y, outputs_rounded.cpu())
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        val_mccs.append(val_mcc)
    print(f'Epoch {e}: train_acc {avg_train_acc} | val_acc {val_acc}')

In [ ]:
# Infer test set
model.eval()
with torch.no_grad():
    preds = model(test_set.X)
    preds_rounded = torch.round(preds)
    test_loss = round(criterion(preds, test_set.y).item(), 4)
    test_acc = round(accuracy_score(test_set.y, preds_rounded), 4)
    test_mcc = round(matthews_corrcoef(test_set.y, preds_rounded), 4)


In [ ]:
# Save hyperaparameters + metrics
model_config = {'aaindex': standardize, 'structure': str(model).replace('\n', ' '), 'optim': type(optimizer).__name__, 'lr': lr, 'loss_fn': type(criterion).__name__, 'activation': type(activation).__name__, 'batch_size': batch_size, 'epochs': n_epochs, 'test_loss': test_loss, 'test_acc': test_acc, 'test_mcc': test_mcc}
temp_path = Path(f'../results/performance_eval/nn_hyperparameter_comparison/v2_aaindex/')
run_nr = len(list(temp_path.iterdir()))
out_folder = temp_path / f'run_{run_nr}'
out_folder.mkdir()
out_file = out_folder / f'{out_folder.stem}_config.txt'
with open(out_file, 'w') as f:
    yaml.dump(model_config, f, sort_keys=False)

epochs = range(1, n_epochs + 1)
plt.plot(epochs, train_losses, label='Training loss')
plt.plot(epochs, val_losses, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig(out_folder / f'{out_folder.stem}_losses.png')
plt.close()

plt.plot(epochs, train_accs, label='Training accuracy')
plt.plot(epochs, val_accs, label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.savefig(out_folder / f'{out_folder.stem}_acc.png')
plt.close()

plt.plot(epochs, train_mccs, label='Training MCC')
plt.plot(epochs, val_mccs, label='Validation MCC')
plt.xlabel('Epoch')
plt.ylabel('MCC')
plt.legend()
plt.grid(True)
plt.savefig(out_folder / f'{out_folder.stem}_mcc.png')
plt.close()